## Solución analítica para un pulso de inyección en un sistema binario: 
sección 3.3 de [M. De Simoni, J. Carrera and X. Sánchez-Vila, A. Guadagnini, 2005] 
WATER RESOURCES RESEARCH, VOL. 41, W11410, doi:10.1029/2005WR004056, 2005

## Objetivo: Visualizar la solución analítica para un pulso de inyección en un sistema binario, y compararlo con las soluciones numéricas.

Consideramos un medio poroso homogéneo 3D con porosidad constante $n$, un flujo uniforme en la dirección $x$, con velocidad del fluido $ V=\frac{q}{n} $, tensor de dispersión diagonal con dispersividad longitudinal $D_L$ y transversal $D_T$.

Se inyecta de manera puntual un volumen $V_e$, que contiene dos especies disueltas con concentraciones $c_1$ y $c_2$.

Las condiciones de frontera son: lejos del punto de inyección las concentraciones tienden a

$
c_i(x \rightarrow \infty , t)=c_{i0}, \qquad i=1,2
$

La condición inicial se expresa como

$
c_i(x,t=0)=\frac{c_{ie}V_e}{n}\,\delta(x)+c_{i0}
$

El sistema está en equilibrio químico, por lo que

$ c_{10}c_{20}=K,  $ y también $ (c_{10}+c_{1e})(c_{20}+c_{2e})=K $

Se introduce el componente conservativo

$
u=c_1-c_2
$

Las condiciones para $u$ son

$
u(x \rightarrow \infty ,t)=u_0
$

$
u(x,t=0)=\frac{u_eV_e}{n}\,\delta(x)+u_0
$

La solución, de acuerdo con [Domenico and Schwartz, 1997, p. 380] en [De Simoni, J. Carrera and X. Sánchez-Vila, A. Guadagnini, 2005] es:

$$ u({x}, t) = u_0 + \frac{18}{\sqrt{2\pi}}
\frac{u_e}{\tilde V_d}
\exp\left(-\frac{1}{2}\rho^2\right)  \tag{26} $$

$$\rho = \sqrt{ \frac{(x - V t)^2}{2 t D_L} + \frac{y^2 + z^2}{2 t D_T} }.
\tag{27} $$

$$\tilde{V}_{d} = \frac{72 \pi n \sqrt{2} t^{3/2} \sqrt{D_L} D_T}{V_e}.
\tag{28}$$

Sustituyendo (27) y (28) en (26) y considerando que el problema es 1D (solo X), se obtiene:
$$u({x}, t) = u_0 +
\frac{u_e\,V_e}{8\,n\,\pi^{3/2}\,t^{3/2}\,\sqrt{D_L}\,D_T}
\exp\!\left(
-\frac{(x - V t)^2}{4 t D_L}
\right). \tag{1}
 $$

Que puede ser expresada como:

$$
u(x,t)=u_0+\frac{M(t)}{\sqrt{4\pi D_L t}}\exp\!\left(-\frac{(x-x_0-Vt)^2}{4D_L t}\right), \hspace{10mm} M(t)=\frac{u_e\,V_e}{4n\pi D_T\,t} = M0
$$


## Entonces, la solución analítica (1) puede ser expresada como:

$$u(x,t) = u_0 + \frac{M0}{\sqrt{4\pi D_L t}}
\exp\!\left(
-\frac{(x - x_0 - V t)^2}{4 D_L t}
\right)  \tag{2}
 $$
 $$u(x,t) = u_0 + \frac{M(t)}{\sqrt{4\pi D_L t}}
\exp\!\left(
-\frac{(x - x_0 - V t)^2}{4 D_L t}
\right)  \tag{3}
 $$

In [ ]:
# Funcion u(x,t) con M(t)
# =========================================================
# 1. Librerías
# =========================================================
import math
import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display

# =========================================================
# 2. Malla espacial (centros de celda)
# =========================================================
Nx = 30
x_vals = np.array([i - 0.5 for i in range(1, Nx + 1)], dtype=float)

t_vals = [0, 2, 10, 20, 30, 40]

# =========================================================
# 3. Parámetros físicos
# =========================================================
ue = 1.0
u0 = 1.0
V  = 0.4
Ve = 0.4
DL = 0.4
DT = 0.2
n  = 0.5

x0 = 10.5          # centro del pulso
M0 = 199.0         # "masa" inicial discreta en t=0 (impulso)

# =========================================================
# 4. Función u(x,t)
# =========================================================
def u_sol(x, t):
    # --- Condición inicial discreta en t=0 ---
    if t == 0:
        u = np.full_like(x, u0, dtype=float)
        # poner el impulso exactamente en x=10.5 (si existe) o en la celda más cercana
        idx = int(np.argmin(np.abs(x - x0)))
        u[idx] = u0 + M0
        return u

    # --- Para t>0: tu ecuación (2) ---
    M_t = (ue * Ve ) / (4 * n * math.pi * DT * t)  # M(t)

    pref = M_t / math.sqrt(4 * math.pi * DL * t)
    #pref = M0 / math.sqrt(4 * math.pi * DL * t)
    expo = np.exp(-((x - x0 - V*t)**2) / (4 * DL * t))

    return u0 + pref * expo

# =========================================================
# 5. Precalcular soluciones
# =========================================================
u_pulso = {t: u_sol(x_vals, t) for t in t_vals}

# =========================================================
# 6. Gráfica interactiva
# =========================================================
estado = {t: False for t in t_vals}
out = widgets.Output()

def actualizar():
    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7,5))

        for t, on in estado.items():
            if on:
                ax.plot(x_vals, u_pulso[t], marker="o", label=f"t = {t}")

        ax.set_xlabel("x")
        ax.set_ylabel("u(x,t)")
        ax.set_title("u(x,t) con impulso inicial en t=0")
        ax.grid(True)
        if any(estado.values()):
            ax.legend()
        plt.show()

def toggle(change, t):
    estado[t] = change["new"]
    actualizar()

botones = []
for t in t_vals:
    b = widgets.ToggleButton(description=f"t = {t}",
                             layout=widgets.Layout(width="90px"))
    b.observe(lambda c, t=t: toggle(c, t), names="value")
    botones.append(b)

display(widgets.VBox([
    widgets.HTML("<b>Selecciona tiempos:</b>"),
    widgets.HBox(botones),
    out
]))




In [ ]:
# VISUALIZACION DE RESULTADOS ANALITICOS Y NUMERICOS 
# =========================================================
# LECTURA DE EXCEL + VISUALIZACIÓN MULTI-SOLUCIONES
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# =========================================================
# 1. Leer archivo Excel
# =========================================================
archivo = "solana1.xlsx"

# Leer hoja completa sin encabezados
df = pd.read_excel(archivo, header=None)

# =========================================================
# 2. Coordenada x (E50:E79 → filas 49:79, columna 4)
# =========================================================
x_excel = pd.to_numeric(df.iloc[49:79, 4], errors="coerce").values

# =========================================================
# 3. SAJJ → F50:J79  (t=2,10,20,30,40)
# =========================================================
SAJJ = {
    2:  pd.to_numeric(df.iloc[49:79, 5], errors="coerce").values,
    10: pd.to_numeric(df.iloc[49:79, 6], errors="coerce").values,
    20: pd.to_numeric(df.iloc[49:79, 7], errors="coerce").values,
    30: pd.to_numeric(df.iloc[49:79, 8], errors="coerce").values,
    40: pd.to_numeric(df.iloc[49:79, 9], errors="coerce").values
}

# =========================================================
# 4. SNJJ → K50:N79 (t=10,20,30,40)
# =========================================================
SNJJ = {
    10: pd.to_numeric(df.iloc[49:79,10], errors="coerce").values,
    20: pd.to_numeric(df.iloc[49:79,11], errors="coerce").values,
    30: pd.to_numeric(df.iloc[49:79,12], errors="coerce").values,
    40: pd.to_numeric(df.iloc[49:79,13], errors="coerce").values
}

# =========================================================
# 5. SNBN → O50:R79 (t=10,20,30,40)
# =========================================================
SNBN = {
    10: pd.to_numeric(df.iloc[49:79,14], errors="coerce").values,
    20: pd.to_numeric(df.iloc[49:79,15], errors="coerce").values,
    30: pd.to_numeric(df.iloc[49:79,16], errors="coerce").values,
    40: pd.to_numeric(df.iloc[49:79,17], errors="coerce").values
}

# =========================================================
# 6. Estados independientes para cada grupo
# =========================================================
estado_u    = {t: False for t in u_pulso}
estado_SAJJ = {t: False for t in SAJJ}
estado_SNJJ = {t: False for t in SNJJ}
estado_SNBN = {t: False for t in SNBN}

out = widgets.Output()

# =========================================================
# 7. Función de actualización de gráfica
# =========================================================
def actualizar():
    with out:
        out.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8,5))

        # u_pulso (tu solución)
        for t, on in estado_u.items():
            if on:
                ax.plot(x_vals, u_pulso[t], "-o", label=f"u_pulso t={t}")

        # SAJJ
        for t, on in estado_SAJJ.items():
            if on:
                ax.plot(x_excel, SAJJ[t], "--s", label=f"SAJJ t={t}")

        # SNJJ
        for t, on in estado_SNJJ.items():
            if on:
                ax.plot(x_excel, SNJJ[t], ":^", label=f"SNJJ t={t}")

        # SNBN
        for t, on in estado_SNBN.items():
            if on:
                ax.plot(x_excel, SNBN[t], "-.d", label=f"SNBN t={t}")

        ax.set_xlabel("x")
        ax.set_ylabel("u(x,t)")
        #ax.set_title("SA-PULSO   vs   SA-JJ")
        #ax.set_title("SA-PULSO   vs   SN-JJ")
        #ax.set_title("SA-PULSO   vs   SN-BN")
        ax.set_title("SN-JJ   vs   SN-BN")
        ax.grid(True)

        if any(estado_u.values()) or any(estado_SAJJ.values()) \
           or any(estado_SNJJ.values()) or any(estado_SNBN.values()):
            ax.legend()

        plt.show()

# =========================================================
# 8. Botones en matriz (tiempos en columnas)
# =========================================================

# todos los tiempos posibles
todos_t = sorted(set(list(u_pulso) + list(SAJJ) + list(SNJJ) + list(SNBN)))

def crear_boton(estado, t):
    if t not in estado:
        return widgets.Label("")  # celda vacía si no existe

    b = widgets.ToggleButton(
        description="",
        value=False,
        layout=widgets.Layout(
            width="40px",
            height="30px",
            border="2px solid black"   # ← marco visible
        ),
        style={"button_color": "white"}  # fondo blanco
    )

    def on_change(change, t=t):
        estado[t] = change["new"]
        actualizar()

    b.observe(on_change, names="value")
    return b


# =========================================================
# 9. Construir tabla de botones
# =========================================================

grid = []

# encabezado
header = [widgets.Label("tipo")] + [widgets.Label(f"t={t}") for t in todos_t]
grid.extend(header)

# filas
filas = [
    ("u_pulso", estado_u),
    ("SAJJ", estado_SAJJ),
    ("SNJJ", estado_SNJJ),
    ("SNBN", estado_SNBN),
]

for nombre, estado in filas:
    grid.append(widgets.Label(nombre))
    for t in todos_t:
        grid.append(crear_boton(estado, t))


# layout tipo tabla
tabla_botones = widgets.GridBox(
    grid,
    layout=widgets.Layout(
        grid_template_columns=f"120px repeat({len(todos_t)}, 60px)",
        grid_gap="5px 5px"
    )
)

display(widgets.VBox([tabla_botones, out]))


## 